In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import sys
import os
sys.path.append(os.path.abspath("../Code"))
import vectorize
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('../Data/raw/tcga_simple_train.csv')

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['t'], test_size=0.2, random_state=23)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=33)

m_train,v = vectorize.vectorizacionBinaria(X_train)
m_test = v.transform(X_test)
m_val = v.transform(X_val)

le = LabelEncoder()
y_train_num = le.fit_transform(y_train)
y_val_num = le.transform(y_val)
y_test_num = le.transform(y_test)

MAX_LEN = 200


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Dropout, LSTM
model = Sequential([
    
    Embedding(input_dim=len(v.get_feature_names_out()) + 1, output_dim=16, input_length=MAX_LEN),


    LSTM(units=8, 
              activation='tanh', 
              dropout=0.6,
              recurrent_dropout=0),

    Dense(4,activation='softmax')
])

model.compile(optimizer="adam", loss='sparse_categorical_crossentropy', metrics=['accuracy'])

c:\Users\tcspo\Desktop\Codigo_Clases\PLN1\practica\OncoNLP\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [22]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_map = v.vocabulary_
index_mapping = {word: (idx + 1) for word, idx in vocab_map.items()} # 0 es padding

def texto_a_secuencia(serie_texto, mapping, max_len):
    sequencias = []
    for texto in serie_texto:

        seq = [mapping[w] for w in texto.split() if w in mapping]
        sequencias.append(seq)

    return pad_sequences(sequencias, maxlen=max_len)
X_train_rnn = texto_a_secuencia(X_train, index_mapping, MAX_LEN)
X_val_rnn = texto_a_secuencia(X_val, index_mapping, MAX_LEN)

In [23]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train_rnn, y_train_num,
    epochs=30,
    batch_size=64,
    validation_data=(X_val_rnn, y_val_num)
)

Epoch 1/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.3361 - loss: 1.3683 - val_accuracy: 0.3208 - val_loss: 1.3492
Epoch 2/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.3458 - loss: 1.3192 - val_accuracy: 0.3257 - val_loss: 1.3094
Epoch 3/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.3976 - loss: 1.2733 - val_accuracy: 0.3947 - val_loss: 1.2773
Epoch 4/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.4397 - loss: 1.2275 - val_accuracy: 0.4249 - val_loss: 1.2562
Epoch 5/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.4776 - loss: 1.1833 - val_accuracy: 0.4165 - val_loss: 1.2482
Epoch 6/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.4958 - loss: 1.1456 - val_accuracy: 0.4262 - val_loss: 1.2457
Epoch 7/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.5061 - loss: 1.1262 - val_accuracy: 0.4225 - val_loss: 1.2510
Epoch 8/30
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.5276 - loss: 1.0875 - val_accuracy: 0.4177 - v

In [25]:
X_test_rnn = texto_a_secuencia(X_test, index_mapping, MAX_LEN)

predicciones_prob = model.predict(X_test_rnn)
clases_predichas = np.argmax(predicciones_prob, axis=1)

print(classification_report(y_test_num,clases_predichas))

33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
              precision    recall  f1-score   support

           0       0.41      0.46      0.43       247
           1       0.50      0.49      0.49       354
           2       0.48      0.55      0.51       310
           3       0.30      0.14      0.19       121

    accuracy                           0.46      1032
   macro avg       0.42      0.41      0.41      1032
weighted avg       0.45      0.46      0.45      1032

